<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day1_a_(260401)_%EA%B3%A0%EA%B8%89_%ED%95%B4%EC%8B%9C%26server_js_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 해시 - 탐색 속도 비교

In [26]:
%%writefile /content/app.js

const fs = require('fs');
const path = require('path');

const FILE_PATH = path.join(__dirname, 'users.csv');
const TARGET_ID = 'user0300000';
const SEARCH_REPEAT = 10;

function readCsvLines() {
    if (!fs.existsSync(FILE_PATH)) {
        throw new Error('users.csv 파일이 없습니다.');
    }

    const content = fs.readFileSync(FILE_PATH, 'utf8'); // 한글 깨짐 방지
    return content.trim().split('\n');
}

function parseCsvToArray() {
    const lines = readCsvLines();
    const header = lines[0].split(',');

    const rows = [];

    for (let i = 1; i < lines.length; i++) {
        const cols = lines[i].split(',');
        rows.push({
            [header[0]]: cols[0],
            [header[1]]: cols[1],
            [header[2]]: Number(cols[2]),
            [header[3]]: cols[3]
        });
    }

    return rows;
}

function buildHashIndex(rows) {
    const index = Object.create(null);

    for (let i = 0; i < rows.length; i++) {
        index[rows[i].user_id] = rows[i];
    }

    return index;
}

function linearSearchFromFile(targetId) {
    const lines = readCsvLines();

    for (let i = 1; i < lines.length; i++) {
        const cols = lines[i].split(',');

        if (cols[0] === targetId) {
            return {
                user_id: cols[0],
                name: cols[1],
                age: Number(cols[2]),
                city: cols[3]
            };
        }
    }

    return null;
}

function linearSearchFromRows(rows, targetId) {
    for (let i = 0; i < rows.length; i++) {
        if (rows[i].user_id === targetId) {
            return rows[i];
        }
    }

    return null;
}

function hashSearch(index, targetId) {
    return index[targetId] || null;
}

function measureIndexBuildCost(rows) {
    const start = process.hrtime.bigint();
    const index = buildHashIndex(rows);
    const end = process.hrtime.bigint();

    console.log(`해시 인덱스 생성 시간: ${Number(end - start) / 1000000} ms`);
    return index;
}

function measureSingleSearch(rows, index) {
    let start;
    let end;
    let result;

    start = process.hrtime.bigint();
    result = linearSearchFromFile(TARGET_ID);
    end = process.hrtime.bigint();
    console.log(`파일 직접 선형 탐색 시간: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    linearSearchFromRows(rows, TARGET_ID);
    end = process.hrtime.bigint();
    console.log(`메모리 배열 선형 탐색 시간: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    hashSearch(index, TARGET_ID);
    end = process.hrtime.bigint();
    console.log(`해시 인덱스 탐색 시간: ${Number(end - start) / 1000000} ms`);

    if (result) {
        console.log(`\n[대상 ID 정보]`);
        console.log(`- ID: ${result.user_id}`);
        console.log(`- 이름: ${result.name}`);
        console.log(`- 나이: ${result.age}`);
        console.log(`- 도시: ${result.city}`);
    } else {
        console.log(`\n[대상 ID 정보를 찾을 수 없습니다: ${TARGET_ID}]`);
    }
}

function measureRepeatedSearch(rows, index) {
    let start;
    let end;

    start = process.hrtime.bigint();
    for (let i = 0; i < SEARCH_REPEAT; i++) {
        linearSearchFromFile(TARGET_ID);
    }
    end = process.hrtime.bigint();
    console.log(`파일 직접 선형 탐색 ${SEARCH_REPEAT}회: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    for (let i = 0; i < SEARCH_REPEAT; i++) {
        linearSearchFromRows(rows, TARGET_ID);
    }
    end = process.hrtime.bigint();
    console.log(`메모리 배열 선형 탐색 ${SEARCH_REPEAT}회: ${Number(end - start) / 1000000} ms`);

    start = process.hrtime.bigint();
    for (let i = 0; i < SEARCH_REPEAT; i++) {
        hashSearch(index, TARGET_ID);
    }
    end = process.hrtime.bigint();
    console.log(`해시 인덱스 탐색 ${SEARCH_REPEAT}회: ${Number(end - start) / 1000000} ms`);
}

function main() {
    const rows = parseCsvToArray();
    console.log(`총 레코드 수: ${rows.length}`);

    const index = measureIndexBuildCost(rows);
    measureSingleSearch(rows, index);
    measureRepeatedSearch(rows, index);
}

main();

Overwriting /content/app.js


In [27]:
!node app.js

총 레코드 수: 300000
해시 인덱스 생성 시간: 269.131537 ms
파일 직접 선형 탐색 시간: 470.667028 ms
메모리 배열 선형 탐색 시간: 5.884448 ms
해시 인덱스 탐색 시간: 0.063899 ms

[대상 ID 정보]
- ID: user0300000
- 이름: 윤하은
- 나이: 20
- 도시: Seoul
파일 직접 선형 탐색 10회: 2815.661606 ms
메모리 배열 선형 탐색 10회: 45.76452 ms
해시 인덱스 탐색 10회: 0.007547 ms


In [32]:
%%writefile /content/app.js

const fs = require('fs');
const path = require('path');

const FILE_PATH = path.join(__dirname, 'logs.csv');
const TARGET_ID = 'user000021';
const SEARCH_REPEAT = 10;

function readCsvLines() {
    if (!fs.existsSync(FILE_PATH)) {
        throw new Error('logs.csv 파일이 없습니다.');
    }

    const content = fs.readFileSync(FILE_PATH, 'utf8'); // 한글 파일 읽기 위함
    return content.trim().split('\n');
}

function parseCsvToArray() {  // csv 파일 내용을 배열로 만드는 함수
    const lines = readCsvLines(); // csv 파일을 읽는 함수
    const header = lines[0].split(','); // user_id,event_type,timestamp,ip

    const rows = [];

    for (let i = 1; i < lines.length; i++) {
        const cols = lines[i].split(','); // ,를 기준으로 데이터 나누어 저장
        rows.push({
            [header[0]]: cols[0],   // user_id
            [header[1]]: cols[1],   // event_type
            [header[2]]: cols[2],   // timestamp
            [header[3]]: cols[3]    // ip
        });
    }

    return rows; // 배열 반환
}

function buildHashIndex(rows) { // 해시 인덱스 생성
    const index = Object.create(null);

    for (let i = 0; i < rows.length; i++) {
        index[rows[i].user_id] = rows[i];
    }

    return index;
}

function linearSearchFromFile(targetId) { // csv 파일에서 선형 탐색 (파일 직접 선형 탐색)
    const lines = readCsvLines();

    for (let i = 1; i < lines.length; i++) {
        const cols = lines[i].split(',');

        if (cols[0] === targetId) { // 타겟 아이디에 해당하는 유저 정보 반환
            return {
                user_id: cols[0],
                event_type: cols[1],
                timestamp: cols[2],
                ip: cols[3]
            };
        }
    }

    return null;
}

function linearSearchFromRows(rows, targetId) { // 선형 탐색 O(n)
    for (let i = 0; i < rows.length; i++) {
        if (rows[i].user_id === targetId) {
            return rows[i];
        }
    }

    return null;
}

function hashSearch(index, targetId) { // 해시 탐색 O(1)
    return index[targetId] || null;
}

function measureIndexBuildCost(rows) {
    const start = process.hrtime.bigint();
    const index = buildHashIndex(rows);
    const end = process.hrtime.bigint();

    console.log(`해시 인덱스 생성 시간: ${Number(end - start) / 1000000} ms`); // 탐색 후 시간차를 기준으로 탐색 시간 출력
    return index;
}

function measureSingleSearch(rows, index) { // 단일 탐색 성능 측정
    let start;
    let end;

    start = process.hrtime.bigint(); // 고정밀 시간(나노초 단위)을 BigInt 타입으로 반환
    linearSearchFromFile(TARGET_ID); // 찾으려는 아이디를 기준으로 선형 탐색 후 유저 정보 반환
    end = process.hrtime.bigint();
    console.log(`파일 직접 선형 탐색 시간: ${Number(end - start) / 1000000} ms`); // 탐색 후 시간차를 기준으로 탐색 시간 출력

    start = process.hrtime.bigint();
    linearSearchFromRows(rows, TARGET_ID);
    end = process.hrtime.bigint();
    console.log(`메모리 배열 선형 탐색 시간: ${Number(end - start) / 1000000} ms`); // 탐색 후 시간차를 기준으로 탐색 시간 출력

    start = process.hrtime.bigint();
    hashSearch(index, TARGET_ID); // 인덱스를 기준으로 해시 탐색
    end = process.hrtime.bigint();
    console.log(`해시 인덱스 탐색 시간: ${Number(end - start) / 1000000} ms`); // 탐색 후 시간차를 기준으로 탐색 시간 출력
}

function measureRepeatedSearch(rows, index) { // 반복 탐색 성능 측정
    let start;
    let end;

    start = process.hrtime.bigint(); // 고정밀 시간(나노초 단위)을 BigInt 타입으로 반환
    for (let i = 0; i < SEARCH_REPEAT; i++) { // SEARCH_REPEAT 수 만큼 반복해서 탐색
        linearSearchFromFile(TARGET_ID);
    }
    end = process.hrtime.bigint();
    console.log(`파일 직접 선형 탐색 ${SEARCH_REPEAT}회: ${Number(end - start) / 1000000} ms`); // 탐색 후 시간차를 기준으로 탐색 시간 출력

    start = process.hrtime.bigint();
    for (let i = 0; i < SEARCH_REPEAT; i++) {
        linearSearchFromRows(rows, TARGET_ID);
    }
    end = process.hrtime.bigint();
    console.log(`메모리 배열 선형 탐색 ${SEARCH_REPEAT}회: ${Number(end - start) / 1000000} ms`); // 탐색 후 시간차를 기준으로 탐색 시간 출력

    start = process.hrtime.bigint();
    for (let i = 0; i < SEARCH_REPEAT; i++) {
        hashSearch(index, TARGET_ID);
    }
    end = process.hrtime.bigint();
    console.log(`해시 인덱스 탐색 ${SEARCH_REPEAT}회: ${Number(end - start) / 1000000} ms`); // 탐색 후 시간차를 기준으로 탐색 시간 출력
}

function main() { // CSV 읽고 해시 인덱스 생성, 단일/반복 탐색 시간 측정
    const rows = parseCsvToArray(); // csv 파일 내부의 값들을 rows라는 배열로 저장
    console.log(`총 로그 수: ${rows.length}`); // rows의 길이(총 로그 수) 출력

    const index = measureIndexBuildCost(rows); // rows의 해시 인덱스 생성 시간을 저장
    measureSingleSearch(rows, index); // 탐색을 한번만 할 경우
    measureRepeatedSearch(rows, index); // 탐색을 10번 반복할 경우
}

main(); // main함수 실행

Overwriting /content/app.js


기본

*   파일 직접 탐색: 느림 (매번 파일에서 직접 읽어봐서 느림)
*   메모리 배열 선형 탐색: 빠름 (배열 뒤쪽의 데이터를 찾을수록 느려짐)
*   해시 탐색: 제일 빠름 (해시 인덱스 생성은 시간이 걸리나 탐색은 빠름)






In [33]:
!node app.js

총 로그 수: 100000
해시 인덱스 생성 시간: 80.302179 ms
파일 직접 선형 탐색 시간: 43.224556 ms
메모리 배열 선형 탐색 시간: 0.119384 ms
해시 인덱스 탐색 시간: 0.046174 ms
파일 직접 선형 탐색 10회: 159.618321 ms
메모리 배열 선형 탐색 10회: 0.02105 ms
해시 인덱스 탐색 10회: 0.007581 ms


맨 마지막

In [31]:
!node app.js

총 로그 수: 100000
해시 인덱스 생성 시간: 81.614633 ms
파일 직접 선형 탐색 시간: 134.384541 ms
메모리 배열 선형 탐색 시간: 3.071239 ms
해시 인덱스 탐색 시간: 0.058808 ms
파일 직접 선형 탐색 10회: 892.940903 ms
메모리 배열 선형 탐색 10회: 9.867785 ms
해시 인덱스 탐색 10회: 0.008431 ms


In [1]:
%%writefile app.js

const fs = require('fs');
const path = require('path');
const mysql = require('mysql2/promise');

const FILE_PATH = path.join(__dirname, 'videos.csv');
const TARGET_ID = 'video000123';

const pool = mysql.createPool({
    host: 'localhost',
    user: 'testuser',
    password: '1234',
    database: 'testdb'
});

function readCsvLines() {
    if (!fs.existsSync(FILE_PATH)) {
        throw new Error('videos.csv 파일이 없습니다.');
    }

    const content = fs.readFileSync(FILE_PATH, 'utf8');
    return content.trim().split('\n');
}

function parseCsvToArray() {
    const lines = readCsvLines();
    const rows = [];

    for (let i = 1; i < lines.length; i++) {
        const cols = lines[i].split(',');
        rows.push({
            video_id: cols[0],
            file_path: cols[1],
            file_hash: cols[2]
        });
    }

    return rows;
}

function buildHashIndex(rows) {
    const index = Object.create(null);

    for (let i = 0; i < rows.length; i++) {
        index[rows[i].video_id] = rows[i];
    }

    return index;
}

function hashSearch(index, targetId) {
    return index[targetId] || null;
}

async function getVideoMetaFromDB(videoId) {
    const sql = `
        SELECT video_id, title, channel_name, views, created_at, status
        FROM video_metadata
        WHERE video_id = ?
    `;

    const [rows] = await pool.execute(sql, [videoId]);
    return rows.length > 0 ? rows[0] : null;
}

async function main() {
    const csvRows = parseCsvToArray();
    const index = buildHashIndex(csvRows);

    const videoFile = hashSearch(index, TARGET_ID);

    if (!videoFile) {
        console.log('영상 파일을 찾지 못했습니다.');
        return;
    }

    console.log('영상 파일 정보');
    console.log(videoFile);

    const videoMeta = await getVideoMetaFromDB(videoFile.video_id);

    if (!videoMeta) {
        console.log('영상 메타데이터를 찾지 못했습니다.');
        return;
    }

    console.log('영상 서비스 정보');
    console.log(videoMeta);

    const mergedResult = {
        video_id: videoFile.video_id,
        file_path: videoFile.file_path,
        file_hash: videoFile.file_hash,
        title: videoMeta.title,
        channel_name: videoMeta.channel_name,
        views: videoMeta.views,
        created_at: videoMeta.created_at,
        status: videoMeta.status
    };

    console.log('결합 출력 결과');
    console.log(mergedResult);

    await pool.end();
}

main().catch(function (err) {
    console.error(err);
});

Writing app.js


In [2]:
!node app.js

영상 파일 정보
{
  video_id: 'video000123',
  file_path: '/data/videos/video000123.mp4',
  file_hash: '437373784e8c720d923a568124fa757cb2bf10d7db86964874f059ae38c99453'
}
영상 서비스 정보
{
  video_id: 'video000123',
  title: 'Node.js 비동기 입문',
  channel_name: '코딩연구소',
  views: 12540,
  created_at: 2026-03-20T09:00:00.000Z,
  status: 'PUBLIC'
}
결합 출력 결과
{
  video_id: 'video000123',
  file_path: '/data/videos/video000123.mp4',
  file_hash: '437373784e8c720d923a568124fa757cb2bf10d7db86964874f059ae38c99453',
  title: 'Node.js 비동기 입문',
  channel_name: '코딩연구소',
  views: 12540,
  created_at: 2026-03-20T09:00:00.000Z,
  status: 'PUBLIC'
}


# 금융 트랜잭션 기반 요청 시스템 구조 설계와 롤백 처리

In [3]:
%%writefile /content/templates/index.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>요청 기반 시스템</title>
</head>
<body>
    <input id="name" placeholder="이름 입력">
    <button onclick="send()">전송</button>

    <script>
        async function send() {
            const name = document.getElementById('name').value;

            const res = await fetch('/api/user', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ name })
            });

            const data = await res.json();
            alert(data.message);
        }
    </script>
</body>
</html>

Writing /content/templates/index.html


In [4]:
%%writefile server.js

const express = require('express');
const path = require('path');
const mysql = require('mysql2/promise');

const app = express();
app.use(express.json());

const pool = mysql.createPool({
    host: 'localhost',
    user: 'testuser',
    password: '1234',
    database: 'testdb'
});

app.get('/', (req, res) => {
    res.sendFile(path.join(__dirname, 'templates', 'index.html'));
});

app.post('/api/user', async (req, res) => {
    const name = req.body.name;

    const sql = 'INSERT INTO users(name) VALUES (?)';
    await pool.execute(sql, [name]);

    res.json({ message: '저장 완료' });
});

app.listen(3000);

Writing server.js


In [13]:
%%writefile /content/templates/transfer.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>계좌이체</title>
</head>
<body>
    <input id="from" placeholder="출금 계좌번호">
    <input id="to" placeholder="입금 계좌번호">
    <input id="amount" placeholder="금액">
    <button onclick="transfer()">이체</button>

    <script>
        async function transfer() {
            const from = document.getElementById('from').value;
            const to = document.getElementById('to').value;
            const amount = Number(document.getElementById('amount').value);

            const res = await fetch('/api/transfer', {
                method: 'POST',
                headers: {'Content-Type':'application/json'},
                body: JSON.stringify({ from, to, amount })
            });

            const data = await res.json();
            alert(data.message);
        }
    </script>
</body>
</html>

Writing /content/templates/transfer.html


In [14]:
%%writefile server.js

const express = require('express');
const path = require('path');
const mysql = require('mysql2/promise');

const app = express();
app.use(express.json());

const pool = mysql.createPool({
    host: 'localhost',
    user: 'testuser',
    password: '1234',
    database: 'testdb'
});

app.get('/', (req, res) => {
    res.sendFile(path.join(__dirname, 'templates', 'transfer.html'));
});

app.post('/api/transfer', async (req, res) => {
    const { from, to, amount } = req.body;

    const conn = await pool.getConnection();

    try {
        await conn.beginTransaction();

        const [fromAccount] = await conn.execute(
            'SELECT balance FROM accounts WHERE account_number = ?',
            [from]
        );

        if (fromAccount.length === 0) {
            throw new Error('출금 계좌 없음');
        }

        if (fromAccount[0].balance < amount) {
            throw new Error('잔액 부족');
        }

        await conn.execute(
            'UPDATE accounts SET balance = balance - ? WHERE account_number = ?',
            [amount, from]
        );

        await conn.execute(
            'UPDATE accounts SET balance = balance + ? WHERE account_number = ?',
            [amount, to]
        );

        await conn.execute(
            'INSERT INTO transactions(from_account, to_account, amount) VALUES (?, ?, ?)',
            [from, to, amount]
        );

        await conn.commit();

        res.json({ message: '이체 완료' });

    } catch (err) {
        await conn.rollback();
        res.json({ message: '이체 실패: ' + err.message });

    } finally {
        conn.release();
    }
});

app.listen(3000);

Overwriting server.js


# 금융 시스템의 로그 추적 및 감사(Audit) 시스템 구현

In [22]:
%%writefile templates/transfer.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>감사 로그 포함 계좌이체</title>
</head>
<body>
    <input id="user_id" placeholder="요청 사용자 ID">
    <input id="from" placeholder="출금 계좌번호">
    <input id="to" placeholder="입금 계좌번호">
    <input id="amount" placeholder="금액">
    <button onclick="transfer()">이체</button>
    <button onclick="location.href='/audit'">감사 로그 보기</button>

    <script>
        async function transfer() {
            const user_id = document.getElementById('user_id').value;
            const from = document.getElementById('from').value;
            const to = document.getElementById('to').value;
            const amount = Number(document.getElementById('amount').value);

            const res = await fetch('/api/transfer', {
                method: 'POST',
                headers: { 'Content-Type': 'application/json' },
                body: JSON.stringify({ user_id, from, to, amount })
            });

            const data = await res.json();
            alert(data.message);
        }
    </script>
</body>
</html>

Overwriting templates/transfer.html


In [23]:
%%writefile templates/audit.html

<!DOCTYPE html>
<html lang="ko">
<head>
    <meta charset="UTF-8">
    <title>감사 로그 조회</title>
</head>
<body>
    <h2>감사 로그 목록</h2>
    <button onclick="loadAudit()">조회</button>
    <table border="1">
        <thead>
            <tr>
                <th>번호</th>
                <th>사용자</th>
                <th>요청유형</th>
                <th>출금계좌</th>
                <th>입금계좌</th>
                <th>금액</th>
                <th>상태</th>
                <th>오류메시지</th>
                <th>요청시각</th>
            </tr>
        </thead>
        <tbody id="auditList"></tbody>
    </table>

    <script>
        async function loadAudit() {
            const res = await fetch('/api/audit');
            const data = await res.json();

            const tbody = document.getElementById('auditList');
            tbody.innerHTML = '';

            for (let i = 0; i < data.length; i++) {
                tbody.innerHTML += `
                    <tr>
                        <td>${data[i].id}</td>
                        <td>${data[i].user_id}</td>
                        <td>${data[i].action_type}</td>
                        <td>${data[i].from_account}</td>
                        <td>${data[i].to_account}</td>
                        <td>${data[i].amount}</td>
                        <td>${data[i].status}</td>
                        <td>${data[i].error_message || ''}</td>
                        <td>${data[i].created_at}</td>
                    </tr>
                `;
            }
        }
    </script>
</body>
</html>

Writing templates/audit.html


In [24]:
%%writefile server.js

const express = require('express');
const path = require('path');
const mysql = require('mysql2/promise');

const app = express();
app.use(express.json());

const pool = mysql.createPool({
    host: 'localhost',
    user: 'testuser',
    password: '1234',
    database: 'testdb',
    waitForConnections: true,
    connectionLimit: 10,
    queueLimit: 0,
    dateStrings: true
});

app.get('/', function (req, res) {
    res.sendFile(path.join(__dirname, 'templates', 'transfer.html'));
});

app.get('/audit', function (req, res) {
    res.sendFile(path.join(__dirname, 'templates', 'audit.html'));
});

app.get('/api/audit', async function (req, res) {
    try {
        const sql = `
            SELECT id, user_id, action_type, from_account, to_account, amount, status, error_message, created_at
            FROM audit_logs
            ORDER BY id DESC
        `;
        const [rows] = await pool.execute(sql);
        res.json(rows);
    } catch (err) {
        res.json([]);
    }
});

app.post('/api/transfer', async function (req, res) {
    const user_id = req.body.user_id;
    const from = req.body.from;
    const to = req.body.to;
    const amount = Number(req.body.amount);

    const conn = await pool.getConnection();

    try {
        await conn.beginTransaction();

        await conn.execute(
            `INSERT INTO audit_logs (user_id, action_type, from_account, to_account, amount, status, error_message)
             VALUES (?, 'TRANSFER_REQUEST', ?, ?, ?, 'START', NULL)`,
            [user_id, from, to, amount]
        );

        const [fromRows] = await conn.execute(
            'SELECT balance FROM accounts WHERE account_number = ?',
            [from]
        );

        if (fromRows.length === 0) {
            throw new Error('출금 계좌 없음');
        }

        const [toRows] = await conn.execute(
            'SELECT balance FROM accounts WHERE account_number = ?',
            [to]
        );

        if (toRows.length === 0) {
            throw new Error('입금 계좌 없음');
        }

        if (amount <= 0) {
            throw new Error('이체 금액 오류');
        }

        if (fromRows[0].balance < amount) {
            throw new Error('잔액 부족');
        }

        await conn.execute(
            'UPDATE accounts SET balance = balance - ? WHERE account_number = ?',
            [amount, from]
        );

        await conn.execute(
            'UPDATE accounts SET balance = balance + ? WHERE account_number = ?',
            [amount, to]
        );

        await conn.execute(
            `INSERT INTO transactions (from_account, to_account, amount)
             VALUES (?, ?, ?)`,
            [from, to, amount]
        );

        await conn.execute(
            `INSERT INTO audit_logs (user_id, action_type, from_account, to_account, amount, status, error_message)
             VALUES (?, 'TRANSFER_RESULT', ?, ?, ?, 'SUCCESS', NULL)`,
            [user_id, from, to, amount]
        );

        await conn.commit();

        res.json({ success: true, message: '이체 완료' });
    } catch (err) {
        await conn.rollback();

        try {
            await pool.execute(
                `INSERT INTO audit_logs (user_id, action_type, from_account, to_account, amount, status, error_message)
                 VALUES (?, 'TRANSFER_RESULT', ?, ?, ?, 'FAIL', ?)`,
                [user_id, from, to, amount, err.message]
            );
        } catch (logErr) {
        }

        res.json({ success: false, message: '이체 실패: ' + err.message });
    } finally {
        conn.release();
    }
});

app.listen(3000, function () {
    console.log('실행: http://localhost:3000');
});

Overwriting server.js


# 공통

In [34]:
!npm install express mysql2

⠙⠹⠸⠼⠴⠦⠧⠇
up to date, audited 77 packages in 1s
⠇
⠇24 packages are looking for funding
⠇  run `npm fund` for details
⠇
found 0 vulnerabilities
⠇

In [35]:
!nohup node server.js > server.log 2>&1 &

In [36]:
!tail -n 20 server.log

실행: http://localhost:3000


In [37]:
!npm install -g cloudflared

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
changed 1 package in 2s
⠋

In [38]:
!nohup cloudflared tunnel --url http://localhost:3000 > tunnel.log 2>&1 &

In [39]:
!tail -n 20 tunnel.log


2026-04-01T08:14:20Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-04-01T08:14:20Z INF Requesting new quick Tunnel on trycloudflare.com...


In [40]:
!grep -o "https://[a-zA-Z0-9.-]*\.trycloudflare.com" tunnel.log |tail -n 1

https://residents-ensures-baker-worcester.trycloudflare.com
